# Pitchers Parquet: relievers grouped by team

Uses **`data/stats_tables/pitchers_{season}.parquet`**, written automatically by **`live`** / **`backfill`**, or by **`run --export-stats-dir ...`**.

- **`Team`**: FanGraphs-style abbrev (MLB API path).
- **`is_reliever_season`**: season rule (low GS share, min games); not the same as relief-only FIP in the **games** file (`home_pen_*` / `away_pen_*`).

Set **`SEASON`** below to match your export.

In [8]:
from pathlib import Path
import pandas as pd

_cwd = Path.cwd()
DATA = _cwd / "data" if (_cwd / "data").is_dir() else _cwd.parent / "data"
STATS_DIR = DATA / "stats_tables"

SEASON = 2026  # change to match your export
pitch_path = STATS_DIR / f"pitchers_{SEASON}.parquet"
if not pitch_path.is_file():
    raise FileNotFoundError(
        f"Missing {pitch_path}. Run e.g.:\n"
        f"  python -m app.main live --season {SEASON}\n"
        f"  python -m app.main backfill --years {SEASON}"
    )

print("Using:", pitch_path.resolve())

Using: C:\Users\Austin\baseball\mlb-model\data\stats_tables\pitchers_2026.parquet


In [9]:
pitch = pd.read_parquet(pitch_path)
pitch.columns.tolist()

['id',
 'Name',
 'Team',
 'K%',
 'BB%',
 'xFIP',
 'kbb',
 'gamesStarted',
 'gamesPlayed',
 'is_reliever_season']

In [10]:
if "is_reliever_season" not in pitch.columns:
    raise ValueError("Re-export pitchers with the latest pipeline (is_reliever_season + Team columns).")

rels = pitch[pitch["is_reliever_season"]].dropna(subset=["Team"])
by_team = (
    rels.groupby("Team", dropna=False)
    .agg(
        n_relievers=("Name", "count"),
        reliever_xfip_mean=("xFIP", "mean"),
        reliever_kbb_mean=("kbb", "mean"),
    )
    .sort_values("reliever_xfip_mean")
)
by_team.head(20)

,n_relievers,reliever_xfip_mean,reliever_kbb_mean
Team,,,
NYY,7,2.003780,18.168520
PHI,8,2.477154,22.398238
SDP,8,2.480571,19.346880
COL,6,2.491692,18.357059
ATL,7,2.574790,22.104569
SEA,8,2.582979,22.895442
NYM,8,3.019036,14.896671
MIL,8,3.158958,14.352904
MIA,7,3.187363,13.499562


In [11]:
# Example: one team's reliever rows
TEAM = "LAD"
rels.loc[rels["Team"] == TEAM, ["Name", "xFIP", "kbb", "gamesStarted", "gamesPlayed"]].sort_values("xFIP")

,Name,xFIP,kbb,gamesStarted,gamesPlayed
116,Edwin Díaz,2.300000,23.809524,0,5
104,Will Klein,2.394118,13.636364,0,4
354,Edgardo Henriquez,2.600000,17.647059,0,4
66,Blake Treinen,3.100000,5.882353,0,5
129,Tanner Scott,3.314286,33.333333,0,6
255,Jack Dreyer,3.330769,5.263158,0,5
69,Alex Vesia,3.500000,0.000000,0,5
366,Ben Casparius,6.645455,5.882353,0,4
